# Caso B · 02 ETL bronce → plata para consumo eléctrico

> _Tutorial · Caso de uso: **B — Forecast consumo 24h** · Capa Medallion: **bronce → plata** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Construir las líneas de InfluxDB line protocol para `power_01`, `temperature_outdoor` y `solar_irradiance` desde el subset BDG2 mock, con tags canónicos. Si el stack está disponible, escribir; si no, persistir el fichero `.lp` en `output/`.


## 2. Qué se aprende

- Mapping CSV → topic / tag / variable.
- Construcción eficiente de line protocol (millones de filas).
- Bulk write a InfluxDB con `write_api`.
- Cómo decidir entre `domain_id=bms_buildings` y `bms_classrooms`.


## 3. Contexto del caso de uso

BDG2 son edificios genéricos, así que usamos `domain_id=bms_buildings` y `site_id=bdg2_education`. Cuando lleguen datos del IES Simarro será `domain_id=bms_classrooms`.


## 4. Relación con CENTINELA+

El bucket `telemetry` (raw 14d) recibe estos puntos. Los rollups se generan automáticamente a `telemetry_1h` que es el más usado por ML.


## 5. Relación con Medallion

Bronce (CSV) → **plata** (InfluxDB).


## 6. Datos de entrada

Mismo CSV que el notebook anterior. Si tienes BDG2 completo, sustituye el path.


## 7. Schema CAPTIA esperado

5 tags: `captia_env=dev`, `domain_id=bms_buildings`, `site_id=bdg2_education`, `asset_id=<building_id>`, `variable ∈ {power_01, temperature_outdoor, solar_irradiance}`.


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Cargamos y reformulamos en formato largo (cada fila = un punto).


In [ ]:
csv_path = ROOT / "notebooks" / "_data" / "bdg2_education_subset_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"])
long = df.melt(
    id_vars=["timestamp", "building_id"],
    value_vars=["power_kw", "t_outdoor", "ghi"],
    var_name="csv_var", value_name="value",
)
mapping = {"power_kw": "power_01", "t_outdoor": "temperature_outdoor", "ghi": "solar_irradiance"}
long["variable"] = long["csv_var"].map(mapping)
long.head()


## 10. Exploración paso a paso

Conteo por variable y fechas mín/máx.


In [ ]:
print("Variables:", long["variable"].unique())
print("Rango:", long["timestamp"].min(), "→", long["timestamp"].max())


## 11. Transformación bronce → plata

Construimos un `Path` de salida y escribimos line protocol por chunks.


In [ ]:
out_dir = ROOT / "output" / "case_B"
out_dir.mkdir(parents=True, exist_ok=True)
lp_path = out_dir / "bdg2_subset.lp"

def to_lp(row):
    ts_ns = int(pd.Timestamp(row["timestamp"]).value)
    return build_line_protocol(
        measurement=MEASUREMENT_TELEMETRY,
        tags={
            "captia_env": "dev", "domain_id": "bms_buildings",
            "site_id": "bdg2_education", "asset_id": row["building_id"],
            "variable": row["variable"],
        },
        fields={"value": float(row["value"])},
        timestamp_ns=ts_ns,
    )

# Para clase: solo primeras 1000 filas
sample = long.head(1000).apply(to_lp, axis=1)
lp_path.write_text("\n".join(sample) + "\n", encoding="utf-8")
print(f"Wrote {lp_path.relative_to(ROOT)} ({len(sample)} líneas)")


## 12. Construcción de capa oro

El bucket `telemetry_1h` es el más usado para ML; lo abordaremos en el notebook 03 (features).


## 13. Visualizaciones explicativas

Verificamos que los timestamps y valores reconstruyen la señal original.


In [ ]:
sample_long = long.head(1000)
ax = sample_long.pivot(index="timestamp", columns="variable", values="value").plot(figsize=(10, 3))
ax.set_title("Sample 1000 puntos · 3 variables")
plt.tight_layout()


## 14. Validaciones

Validamos schema y formato de las primeras 5 líneas.


In [ ]:
firstlines = lp_path.read_text(encoding="utf-8").splitlines()[:5]
for ln in firstlines:
    assert ln.startswith("captia_point,")
    assert "captia_env=dev" in ln
    assert "value=" in ln
    print(ln)


## 15. Errores comunes

1. Confundir `,` (separador de tags) con `=`.
2. Olvidar el espacio entre tags y fields.
3. Escribir el timestamp en segundos (Influx 2 espera **ns**).
4. Usar `bool_state` en `domain_id=bms_buildings` (no aplica).


## 16. Ejercicios propuestos

1. Escribe `to_lp_batch(df)` que produzca el line protocol completo.
2. Sube el fichero con `influx write -f bdg2_subset.lp`.
3. Mide el throughput (líneas/s).


## 17. Cómo se reutiliza con datos reales

Para escribir a `simarro-prod`: cambiar `domain_id` y `site_id`. El resto es idéntico.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `02_case_B_energy_forecasting/03_features_forecasting.ipynb`.
- Documento web del caso: `docs/contracts/influx-schema.md`.
